In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
import pickle
import glob

import numpy as np
import scipy
import pandas as pd
from scipy.stats import pearsonr, zscore
from scipy.ndimage import convolve1d
import statsmodels

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import r2_score

import imageio.v2 as imageio
import torch
import torch.nn.functional as F

from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor

import utils
from utils import align_channels_across_runs, compute_ncsnr_all_timepoints, reshape_electrode_data_by_stimuli, ncsnr_figs, do_retrieval
# from PSN.psn import psn

from mne.stats import permutation_cluster_1samp_test
from concurrent.futures import ThreadPoolExecutor

np.random.seed(42)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

assert device == "cuda"

# load data from a subject

In [ ]:
base_dir = '/home/ri99/ieeg'
ieeg_dir = f'{base_dir}/data/subjects/111'
csv_dir = f'{base_dir}/data/nsd_PTB'
img_dir = f'{base_dir}/data/nsd_ecog'

data_dir = f'{base_dir}/data/hfb'
output_dir = f'{base_dir}/outputs/hfb'
cache_dir = f'{base_dir}/.cache'

os.makedirs(cache_dir, exist_ok=True)
sub = 11
n_blocks = -1  # -1 = use all blocks
n_runs = 56

sampling_rate = 1024  # Hz
baseline_time = 0.2  # seconds of data per trial before stimulus onset
# subject 111 had 5600 images split into 4 distinct sessions over 2 days (day 1: runs 1-12 and 13-24; day 2: runs 25-36 and 37-56)

load_from_file = False
smoothing = True

if smoothing:
    window_size = 25
    stride = 10
    smooth_suffix = f'_window{window_size}_stride{stride}'
else:
    smooth_suffix = ''

In [ ]:
curric = pd.read_csv(f"{csv_dir}/curriculum_patient{sub}.csv")
curric.head()

In [ ]:
if n_blocks == -1:  # use all available blocks
    block_list = curric['block'].unique().astype(str)
else:  # use n_blocks to specify instead
    block_list = [str(x) for x in range(1, n_blocks+1)]

print(block_list)
image_names = list(curric[curric['block'].isin(block_list)]['filename'])
print(len(image_names))

In [ ]:
# load bipolar-referenced voltage trace data (no frequency decomposition) at 1024Hz
run_list = list(range(1, n_runs+1))

all_data = []
all_labels = []
time_ref = None

for run in tqdm(run_list):
    if load_from_file is False:
        data = np.load(f"{ieeg_dir}/bandpower1024/npy/sub1{sub}_NSD_run{run}_bandpower70-170.npy")
    labels = pd.read_csv(f"{ieeg_dir}/bandpower1024/ch_labels/sub1{sub}_NSD_run{run}_bandpower_channels.csv")
    time = np.load(f"{ieeg_dir}/bandpower1024/time/sub1{sub}_NSD_run{run}_bandpower_time.npy")

    if load_from_file is False:
        assert time.shape[0] == data.shape[2], f"time mismatch in run {run}"
        assert labels.shape[0] == data.shape[1], f"label mismatch in run {run}"
    
    if time_ref is None:
        time_ref = time
    else:
        assert np.allclose(time, time_ref), f"time array differs in run {run}"

    if load_from_file is False:
        all_data.append(data)
    all_labels.append(labels)

if load_from_file is False:
    print(len(all_data))
    del data

# delete to prevent using the wrong ones later; we use channel_info to index channel labels from here on
del labels

In [ ]:
if load_from_file is False:
    # combine data across runs while dealing with some runs not having data from the same channels
    data, channel_info = align_channels_across_runs(all_data, all_labels)
    data = data.transpose(1,2,0).astype(np.float32)  # convert to channels x time x trials
    print(f"Final shape: {data.shape}")
    print(channel_info.head())
    channel_info.to_csv(f'{data_dir}/channel_info.csv', index=False)
else:
    channel_info = pd.read_csv(f'{data_dir}/channel_info.csv')

del all_labels

# reshape to identify repeated stimuli

In [ ]:
if load_from_file is False:
    # First, reshape your electrode data using the existing function
    reshaped_electrode_data, stim_info = reshape_electrode_data_by_stimuli(
        data,  # Your electrode data (channels, time, trials)
        curric,         # Your events dataframe
        stim_column='coco_id'
    )
    np.save(f'{data_dir}/reshaped_electrode_data', reshaped_electrode_data)
    with open(f'{data_dir}/stim_info.pkl', 'wb') as f:
        pickle.dump(stim_info, f)
else:
    print('loading reduced_data, stim_info from file')
    reduced_data = np.load(f'{output_dir}/top_ncsnr_electrodes_timepoints{smooth_suffix}.npy')
    with open(f'{data_dir}/stim_info.pkl', 'rb') as f:
        stim_info = pickle.load(f)
    print(reduced_data.shape)

In [ ]:
unique_image_indices = [idx[0] for idx in list(stim_info['stimulus_to_trials'].values())]
unique_image_names = pd.unique(curric['filename'])

# sanity checks
assert len(unique_image_indices) == stim_info['n_unique_images']
assert np.all(list(curric['filename'][curric['is_repeat']==0]) == unique_image_names)
assert np.all(list(curric['filename'][curric['repetition_number']==1]) == unique_image_names)

In [ ]:
if load_from_file == False:
    images = utils.load_images(unique_image_names, img_dir=img_dir, save_path=f'{output_dir}/images_torch')
else:
    images = torch.load(f'{output_dir}/images_torch', weights_only=True)

print(images.shape, type(images), images.dtype)
print(images.min(), images.max())

In [ ]:
for i in range(len(unique_image_names)):
    # assert the number of available (not NaN) repetitions in the data matches the expected number from the stimulus ordering
    assert int(np.sum(~np.isnan(reshaped_electrode_data[0,i,0]))) == len(list(stim_info['stimulus_to_trials'].values())[i])

In [ ]:
np.array(stim_info['unique_stimuli'])[multi_rep_idx]

In [ ]:
multi_rep_idx = np.where(np.array(list(stim_info['stimulus_counts'].values())) != 1)[0]  # conditions with >1 repeat

reduced_data = reduced_data[:, multi_rep_idx]  # only conditions with repeats
images = images[multi_rep_idx]

print('removed conditions without repeats')
print(reduced_data.shape)
print(images.shape)

In [ ]:
# cleaned_data = np.nanmean(reduced_data, axis=3)
# print('averaged over repeats')
# print(cleaned_data.shape)
# del reduced_data

In [ ]:
stim_info.keys()

In [ ]:
stim_info['stimulus_counts']

In [ ]:
cleaned_data = reduced_data.transpose(1,0,2,3)

print(f"images shape: {list(images.shape)}, {images.dtype}")
print(f"ieeg data shape: {list(cleaned_data.shape)}")
print(f"ieeg data interpretation: {cleaned_data.shape[0]} conditions, {cleaned_data.shape[1]} channels, {cleaned_data.shape[2]} timepoints")

In [ ]:
# custom split by trial repetitions
# train and test have equal amounts of 

In [ ]:
# train/test split
x_train, x_test, y_train, y_test, train_idx, test_idx = train_test_split(
    cleaned_data, clip_image, np.arange(cleaned_data.shape[0]), test_size=0.20, random_state=42
)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

In [ ]:
# reshape to flatten channels and timepoints
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

print(x_train_flat.shape, x_test_flat.shape, y_train.shape, y_test.shape)

In [ ]:
# no z-scoring at all
x_train_scaled = x_train_flat
x_test_scaled = x_test_flat

# baseline mean subtraction per channel
# take each electrode's baseline signal from -200ms to 0ms
# baseline_data = np.nanmean(reshaped_electrode_data[rankings[:n_reliable_channels], :, :int(np.floor(sampling_rate*0.2))], axis=-1)
# baseline_data = np.delete(baseline_data, 2, axis=0)
# baseline_channel_mean = baseline_data[:,train_idx].mean(axis=1).mean(axis=1)  # per-channel baseline mean for train trials

# x_train_scaled = x_train - baseline_channel_mean.reshape(1,-1,1)
# x_test_scaled = x_test - baseline_channel_mean.reshape(1,-1,1)

# x_train_scaled = x_train_scaled.reshape(x_train_scaled.shape[0], -1)
# x_test_scaled = x_test_scaled.reshape(x_test_scaled.shape[0], -1)

# TODO z-score globally per channel
pass

# z-score per channel per trial
# scaler = StandardScaler()
# x_train_scaled = scaler.fit_transform(x_train_flat)
# x_test_scaled = scaler.transform(x_test_flat)

print(x_train_scaled.shape, x_test_scaled.shape, y_train.shape, y_test.shape)

In [ ]:
from sklearn.decomposition import PCA

# Flatten to (conditions, channels*time)
pca = PCA().fit(x_train_scaled)
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('PC'); plt.ylabel('Cumulative variance')

x = 0.98
plt.axhline(x, c='r', linestyle=':')
n_pcs = np.searchsorted(np.cumsum(pca.explained_variance_ratio_), x) + 1
print(f"{n_pcs} PCs explain {x} of the variance")

plt.show()

In [ ]:
print(x_train_scaled.shape, x_test_scaled.shape, y_train.shape, y_test.shape)

pca = PCA(n_components=n_pcs)
x_train_scaled = pca.fit_transform(x_train_scaled)
x_test_scaled = pca.transform(x_test_scaled)

print(x_train_scaled.shape, x_test_scaled.shape, y_train.shape, y_test.shape)

In [ ]:
# simple retrieval only works if there's no repeats
assert len(cleaned_data) == len(unique_image_names)

In [ ]:
# Set up ridge regression
ridge_model = RidgeCV(alpha_per_target=True)
ridge_model.fit(x_train_scaled, y_train)
y_pred_train = ridge_model.predict(x_train_scaled)
y_pred_test = ridge_model.predict(x_test_scaled)

coef_of_det = ridge_model.score(x_test_scaled, y_test)
print(f'R^2 (coefficient of determination): {coef_of_det:.2f}')

In [ ]:
top_k = 10

train_acc, train_sims = do_retrieval(x_train_scaled, y_train, y_pred_train, top_k=top_k)
test_acc, test_sims = do_retrieval(x_test_scaled, y_test, y_pred_test, top_k=top_k)
print(f"train fwd acc: {train_acc:.2%}, test fwd acc: {test_acc:.2%}")

In [ ]:
# print('displaying every 20 train images and every 5 test images')

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(train_sims[::20, ::20], vmin=-1, vmax=1, cmap="coolwarm", square=True, ax=axs[0])
axs[0].set_title("Train Cosine Similarity Matrix")
axs[0].set_xlabel("Predicted CLIP embedding")
axs[0].set_ylabel("True CLIP embedding")

sns.heatmap(test_sims[::5, ::5], vmin=-1, vmax=1, cmap="coolwarm", square=True, ax=axs[1])
axs[1].set_title("Test Cosine Similarity Matrix")
axs[1].set_xlabel("Predicted CLIP embedding")
axs[1].set_ylabel("True CLIP embedding")

plt.tight_layout()
plt.show()